In [ ]:
!pip install -q openai

In [24]:
import json
import functools

MODEL = "openai/gpt-oss-20b"

MENU = {
    "plum cake": 500,
    "chocolate cake": 650,
    "cupcake": 80
}

SYSTEM = f"""
ROLE:
You are a friendly assistant for Sweet Crumbs Home Bakery.

FACTS:
Menu: {json.dumps(MENU)}
Orders require 24 hours advance notice.
All cakes have eggless versions.

RULES:
Never provide incorrect prices.
Never promise delivery times.
If information is unavailable, say
"I'll check with the owner."

Remember the customer's name and order
using conversation history.

FORMAT:
Keep answers under 50 words.
Reply in English or Telugu matching the customer.
"""

# Decorator
def log_call(func):

    @functools.wraps(func)
    def wrapper(*args, **kwargs):

        result = func(*args, **kwargs)

        print("[LOG]", func.__name__, "->", result)

        return result

    return wrapper


# Price calculation
@log_call
def get_price(item: str, qty: int = 1) -> int:
    """Calculate the total price of bakery products."""

    item = item.lower().strip()

    if item not in MENU:
        raise ValueError("Item not available!")

    if not isinstance(qty, int) or qty < 1:
        raise ValueError("Invalid quantity!")

    return MENU[item] * qty


# Conversation memory
history = [
    {"role": "system", "content": SYSTEM}
]

MAX_TURNS = 8


# Chatbot function
def chat(user_text: str) -> str:

    history.append({
        "role": "user",
        "content": user_text
    })

    recent = [history[0]] + history[1:][-MAX_TURNS:]

    try:

        response = client.chat.completions.create(
            model=MODEL,
            messages=recent,
            temperature=0.3
        )

        reply = response.choices[0].message.content

    except RateLimitError:

        reply = "We are busy. Please try again later."

    except AuthenticationError:

        reply = "API key is not working."

    except APIConnectionError:

        reply = "Connection problem. Try again."

    history.append({
        "role": "assistant",
        "content": reply
    })

    return reply


# Test price calculation
print("Six cupcakes total:", get_price("cupcake", 6))


# Start chatbot
while True:

    user = input("You: ")

    if user.lower().strip() in ["bye", "exit", "quit"]:

        print("Bot: Thank you for visiting Sweet Crumbs!")
        break

    print("Bot:", chat(user))

[LOG] get_price -> 480
Six cupcakes total: 480
You: I want 2 chocolate cakes for tomorrow.
Bot: Sure! 2 chocolate cakes at ₹650 each (₹1,300 total). We’ll have them ready for pickup tomorrow. Please confirm your address and any special requests.
You: Are they eggless?
Bot: Yes, all our cakes—including chocolate cake—are available in eggless versions.
You: What is the price for two?
Bot: ₹1,300 total for the two chocolate cakes.
You: What did I order?
Bot: You ordered 2 chocolate cakes.
You: Can you guarantee delivery at 9 AM?
Bot: I can’t guarantee delivery at 9 AM.
You: bye
Bot: Thank you for visiting Sweet Crumbs!
